In [ ]:
# Basic imports
import numpy as np
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasettfinal/fixed_cleaned_file.jsonl


In [ ]:
pip install trl

In [ ]:
pip install -U bitsandbytes

In [ ]:
import os

# Set single GPU BEFORE importing torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import torch
import gc
import numpy as np
import pandas as pd
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Number of GPUs visible: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("Imports complete")

2025-12-16 17:18:19.672608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765905499.693498     196 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765905499.699814     196 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

PyTorch version: 2.6.0+cu124
CUDA available: True
Number of GPUs visible: 1
GPU: Tesla P100-PCIE-16GB
✅ Imports complete


In [ ]:
CONFIG = {
    "model_name": "mistralai/Mistral-7B-v0.1",
    "data_path": "/kaggle/input/datasettfinal/fixed_cleaned_file.jsonl",
    "output_dir": "/kaggle/working/mistral_finetuned",
    "max_seq_length": 512,
    
    # LoRA Configuration
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
    
    # Training Configuration
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "max_steps": 500,
    "save_steps": 100,
    "logging_steps": 10,
}

print("Configuration loaded")

✅ Configuration loaded


In [ ]:
def check_environment():
    """Check and print environment information"""
    print("=" * 60)
    print("ENVIRONMENT CHECK")
    print("=" * 60)
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    
    if torch.cuda.is_available():
        print(f"CUDA version: {torch.version.cuda}")
        print(f"Number of GPUs: {torch.cuda.device_count()}")
        print(f"GPU 0: {torch.cuda.get_device_name(0)}")
        
        free_mem = torch.cuda.mem_get_info(0)[0] / 1024**3
        total_mem = torch.cuda.mem_get_info(0)[1] / 1024**3
        print(f"Memory: {free_mem:.1f} GB free / {total_mem:.1f} GB total")
    
    print("=" * 60)
    
    # Clear GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    print("GPU memory cleared\n")

check_environment()

ENVIRONMENT CHECK
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Number of GPUs: 1
GPU 0: Tesla P100-PCIE-16GB
Memory: 15.6 GB free / 15.9 GB total
✅ GPU memory cleared



In [ ]:
def fix_jsonl_file(input_path, output_path):
    """Fix and validate JSONL file"""
    print(f"Fixing JSONL file: {input_path}")
    valid_lines = 0
    invalid_lines = 0
    
    with open(input_path, "r", encoding="utf-8") as infile, \
         open(output_path, "w", encoding="utf-8") as outfile:
        
        for line_num, line in enumerate(infile, 1):
            line = line.strip()
            if not line:
                continue
                
            try:
                data = json.loads(line)
                if "prompt" in data and "code" in data:
                    fixed_line = json.dumps(data, ensure_ascii=False)
                    outfile.write(fixed_line + "\n")
                    valid_lines += 1
                else:
                    invalid_lines += 1
            except json.JSONDecodeError:
                invalid_lines += 1
    
    print(f"Valid lines: {valid_lines}")
    print(f"Invalid lines: {invalid_lines}")
    print(f"Fixed file saved to: {output_path}\n")
    
    return output_path

print("Function defined: fix_jsonl_file")

✅ Function defined: fix_jsonl_file


In [ ]:
def load_and_prepare_dataset(data_path):
    """Load and prepare dataset for training"""
    print("Loading dataset...")
    dataset = load_dataset("json", data_files=data_path, split="train")
    
    if "code" in dataset.column_names:
        dataset = dataset.rename_column("code", "completion")
    
    def format_instruction(example):
        text = f"### Prompt:\n{example['prompt']}\n\n### Code:\n{example['completion']}"
        return {"text": text}
    
    dataset = dataset.map(format_instruction, remove_columns=dataset.column_names)
    
    print(f"Dataset size: {len(dataset)}")
    print(f"Sample entry:\n{dataset[0]['text'][:300]}...\n")
    
    return dataset

print("Function defined: load_and_prepare_dataset")

✅ Function defined: load_and_prepare_dataset


In [ ]:
def setup_model_and_tokenizer(model_name, max_seq_length):
    """
    Setup model and tokenizer with 4-bit quantization
    FIXED: Using float16 only (P100 compatible)
    """
    
    # Clear GPU memory
    gc.collect()
    torch.cuda.empty_cache()
    
    print(f"Loading model: {model_name}")
    
    free_mem = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU 0 free memory: {free_mem:.2f} GB")
    
    # Create offload folder
    offload_folder = "/kaggle/working/offload"
    os.makedirs(offload_folder, exist_ok=True)
    
    # Use float16 ONLY - P100 doesn't support bfloat16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,  # MUST be float16
        bnb_4bit_use_double_quant=True,
    )
    
    # Load tokenizer first
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, 
        trust_remote_code=True,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.model_max_length = max_seq_length
    
    # Load model
    print("Loading model with 4-bit quantization (float16)...")
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map={"": 0},  # Force single GPU
        torch_dtype=torch.float16,  # MUST be float16
        trust_remote_code=True,
        low_cpu_mem_usage=True,
    )
    
    # Configure model
    model.config.use_cache = False
    model.config.pad_token_id = tokenizer.pad_token_id
    
    # Prepare for training
    model = prepare_model_for_kbit_training(model)
    
    # Verify no bfloat16 tensors
    print("\nVerifying tensor dtypes...")
    has_bf16 = False
    for name, param in model.named_parameters():
        if param.dtype == torch.bfloat16:
            has_bf16 = True
            print(f"Found bfloat16: {name}")
            break
    
    if not has_bf16:
        print("No bfloat16 tensors found - safe to train!")
    
    print(f"\nModel loaded successfully!")
    print(f"Tokenizer max length: {tokenizer.model_max_length}")
    
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    print(f"GPU 0 memory used: {allocated:.2f} GB\n")
    
    return model, tokenizer

print("Function defined: setup_model_and_tokenizer")

✅ Function defined: setup_model_and_tokenizer


In [ ]:
def setup_lora(model, config):
    """Setup LoRA configuration"""
    print("Setting up LoRA...")
    
    lora_config = LoraConfig(
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config["lora_target_modules"],
    )
    
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    print()
    
    return model

print("Function defined: setup_lora")

✅ Function defined: setup_lora


In [ ]:
def setup_trainer(model, tokenizer, dataset, config):
    """
    Setup SFTTrainer
    FIXED: Disabled mixed precision to avoid AMP issues on P100
    """
    
    print("Setting up trainer...")
    
    from trl import SFTTrainer, SFTConfig
    import trl
    print(f"TRL version: {trl.__version__}")
    
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
    
    # Disable mixed precision to avoid AMP/bfloat16 issues
    print("Using: NO mixed precision (fp16=False, bf16=False)")
    
    sft_config = SFTConfig(
        output_dir=config["output_dir"],
        per_device_train_batch_size=config["per_device_train_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],
        optim="paged_adamw_8bit",
        learning_rate=config["learning_rate"],
        lr_scheduler_type="cosine",
        warmup_steps=50,
        max_steps=config["max_steps"],
        save_steps=config["save_steps"],
        logging_steps=config["logging_steps"],
        
        # CRITICAL: Disable mixed precision
        fp16=False,
        bf16=False,
        
        gradient_checkpointing=True,
        save_total_limit=3,
        report_to="none",
        max_grad_norm=0.3,
        
        # Additional stability settings
        ddp_find_unused_parameters=False,
    )
    
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset,
        processing_class=tokenizer,
        args=sft_config,
    )
    
    print("Trainer setup complete\n")
    return trainer

print("Function defined: setup_trainer")

✅ Function defined: setup_trainer


In [ ]:
def main():
    """Main training pipeline"""
    # Step 1: Check environment
    check_environment()
    
    # Step 2: Fix JSONL file
    fixed_data_path = "/kaggle/working/fixed_cleaned_file.jsonl"
    fix_jsonl_file(CONFIG["data_path"], fixed_data_path)
    
    # Step 3: Load dataset
    dataset = load_and_prepare_dataset(fixed_data_path)
    
    # Step 3b: Split dataset into train/validation/test
    dataset = dataset.train_test_split(test_size=0.2, seed=42)
    val_test = dataset['test'].train_test_split(test_size=0.5, seed=42)
    train_dataset = dataset['train']
    val_dataset = val_test['train']
    test_dataset = val_test['test']
    print(f"Dataset sizes -> train: {len(train_dataset)}, val: {len(val_dataset)}, test: {len(test_dataset)}")
    
    # Step 4: Setup model and tokenizer
    model, tokenizer = setup_model_and_tokenizer(
        CONFIG["model_name"],
        CONFIG["max_seq_length"]
    )
    
    # Step 5: Setup LoRA
    model = setup_lora(model, CONFIG)
    
    # Step 6: Setup trainer
    trainer = setup_trainer(model, tokenizer, train_dataset, CONFIG)
    
    # Step 7: Train
    print("=" * 60)
    print("STARTING TRAINING")
    print("=" * 60)
    history = trainer.train()
    
    # Step 8: Save model
    print("Saving final model...")
    trainer.save_model(CONFIG["output_dir"])
    tokenizer.save_pretrained(CONFIG["output_dir"])
    
    print(f"Training complete. Model saved to: {CONFIG['output_dir']}")
    
    # Optional: collect and plot post-training metrics (call manually if needed)
    # from this notebook: collect_and_plot_post_training(trainer, model, tokenizer, train_dataset, val_dataset, test_dataset, CONFIG['output_dir'])

print("Function defined: main")

✅ Function defined: main


In [ ]:
# Clear GPU before starting
gc.collect()
torch.cuda.empty_cache()

# Run training
if __name__ == "__main__":
    main()

ENVIRONMENT CHECK
PyTorch version: 2.6.0+cu124
CUDA available: True
CUDA version: 12.4
Number of GPUs: 1
GPU 0: Tesla P100-PCIE-16GB
Memory: 15.6 GB free / 15.9 GB total
✅ GPU memory cleared

Fixing JSONL file: /kaggle/input/datasettfinal/fixed_cleaned_file.jsonl
Valid lines: 399
Invalid lines: 0
Fixed file saved to: /kaggle/working/fixed_cleaned_file.jsonl

Loading dataset...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/399 [00:00<?, ? examples/s]

Dataset size: 399
Sample entry:
### Prompt:
Draw a circle with radius 3.5 cm.

### Code:
from manim import *

class DrawCircle(Scene):
    def construct(self):
        radius = 1.37  # 3.5 cm ≈ 1.37 units (1 unit ≈ 2.54 cm)
        circle = Circle(radius=radius, color=BLUE)
        self.play(Create(circle))
        self.wait(2)
...

Loading model: mistralai/Mistral-7B-v0.1
GPU 0 free memory: 15.64 GB
Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


...

Token indices sequence length is longer than the specified maximum sequence length for this model (525 > 512). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/399 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


✅ Trainer setup complete

STARTING TRAINING


Step,Training Loss
10,0.665800
20,0.545900


In [ ]:
# Post-training metrics and plotting utilities
import matplotlib.pyplot as plt
import seaborn as sns
import evaluate
import json
from collections import Counter

rouge_metric = evaluate.load('rouge')
bleu_metric = evaluate.load('sacrebleu')

def compute_generation_metrics(model, tokenizer, dataset, max_new_tokens=128, batch=4):
    predictions = []
    references = []
    model_device = next(model.parameters()).device
    for i in range(0, len(dataset), batch):
        batch_examples = dataset[i:i+batch]
        texts = [ex['text'] for ex in batch_examples]
        inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(model_device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        for j, ex in enumerate(batch_examples):
            predictions.append(decoded[j])
            if 'completion' in ex:
                references.append(ex['completion'])
            else:
                txt = ex['text']
                if '### Code:' in txt:
                    ref = txt.split('### Code:')[-1].strip()
                else:
                    ref = ''
                references.append(ref)
    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    bleu_res = bleu_metric.compute(predictions=predictions, references=[[r] for r in references])
    return {'rouge': rouge_res, 'bleu': bleu_res}

def plot_and_save(metrics_history, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    # 1. training/validation loss
    if 'train_loss' in metrics_history or 'eval_loss' in metrics_history:
        plt.figure(figsize=(8,4))
        if 'train_loss' in metrics_history:
            plt.plot(metrics_history.get('train_loss', []), label='train_loss')
        if 'eval_loss' in metrics_history:
            plt.plot(metrics_history.get('eval_loss', []), label='val_loss')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'loss_curve.png'))
        plt.close()
    # 2. ROUGE bar chart
    if 'rouge' in metrics_history:
        r = metrics_history['rouge']
        items = {k: v for k, v in r.items() if isinstance(v, (int, float))}
        if items:
            plt.figure(figsize=(8,4))
            sns.barplot(x=list(items.keys()), y=list(items.values()))
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, 'rouge_scores.png'))
            plt.close()
    # 3. BLEU bar
    if 'bleu' in metrics_history:
        b = metrics_history['bleu']
        score = b.get('score', b.get('bleu', 0)) if isinstance(b, dict) else 0
        plt.figure(figsize=(4,3))
        plt.bar(['bleu'], [score])
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'bleu_score.png'))
        plt.close()
    # 4. dataset split pie
    if 'splits' in metrics_history:
        sizes = metrics_history['splits']
        plt.figure(figsize=(5,5))
        plt.pie(list(sizes.values()), labels=list(sizes.keys()), autopct='%1.1f%%')
        plt.savefig(os.path.join(output_dir, 'dataset_split.png'))
        plt.close()
    # 5. token length distribution
    if 'token_lengths' in metrics_history:
        lens = metrics_history['token_lengths']
        plt.figure(figsize=(8,4))
        sns.histplot(lens, bins=50)
        plt.xlabel('Token length')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'token_length_dist.png'))
        plt.close()
    # 6. GPU memory
    if 'gpu_mem' in metrics_history:
        plt.figure(figsize=(8,4))
        plt.plot(metrics_history['gpu_mem'])
        plt.xlabel('Checkpoint')
        plt.ylabel('GPU memory (GB)')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'gpu_mem.png'))
        plt.close()
    # 7. parameter counts (trainable vs total)
    if 'param_counts' in metrics_history:
        pc = metrics_history['param_counts']
        plt.figure(figsize=(5,4))
        plt.pie([pc['trainable'], pc['total']-pc['trainable']], labels=['trainable','frozen'], autopct='%1.1f%%')
        plt.savefig(os.path.join(output_dir, 'param_trainable_pie.png'))
        plt.close()
    # 8. learning rate schedule if available
    if 'lr' in metrics_history:
        plt.figure(figsize=(8,4))
        plt.plot(metrics_history['lr'])
        plt.xlabel('Step')
        plt.ylabel('Learning rate')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'lr_schedule.png'))
        plt.close()
    # 9. top tokens (sampled from train)
    if 'top_tokens' in metrics_history:
        toks = metrics_history['top_tokens']
        labels, vals = zip(*toks)
        plt.figure(figsize=(10,4))
        sns.barplot(x=list(labels), y=list(vals))
        plt.xticks(rotation=90)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'top_tokens.png'))
        plt.close()
    # 10. generated length distribution
    if 'gen_lens' in metrics_history:
        plt.figure(figsize=(8,4))
        sns.histplot(metrics_history['gen_lens'], bins=30)
        plt.xlabel('Generated length')
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, 'gen_length_dist.png'))
        plt.close()
    print('Saved charts to', output_dir)

def collect_and_plot_post_training(trainer, model, tokenizer, train_dataset, val_dataset, test_dataset, output_dir):
    metrics_history = {}
    # try to extract loss and lr history from trainer.state.log_history
    try:
        state = getattr(trainer, 'state', None)
        if state is not None and hasattr(state, 'log_history'):
            logs = state.log_history
            train_loss = [x.get('loss') for x in logs if 'loss' in x]
            eval_loss = [x.get('eval_loss') for x in logs if 'eval_loss' in x]
            lr = [x.get('learning_rate') for x in logs if 'learning_rate' in x]
            metrics_history['train_loss'] = train_loss
            metrics_history['eval_loss'] = eval_loss
            metrics_history['lr'] = lr
    except Exception as e:
        print('Could not extract trainer history:', e)
    metrics_history['splits'] = {'train': len(train_dataset), 'val': len(val_dataset), 'test': len(test_dataset)}
    # token lengths sample
    sample_n = min(500, len(train_dataset))
    sample_texts = [ex['text'] for ex in train_dataset.select(range(sample_n))]
    token_lengths = [len(tokenizer(text)['input_ids']) for text in sample_texts]
    metrics_history['token_lengths'] = token_lengths
    # param counts
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    metrics_history['param_counts'] = {'total': total, 'trainable': trainable}
    # top tokens from sample
    tok_ids = []
    for t in sample_texts:
        tok_ids.extend(tokenizer.encode(t, truncation=True, max_length=tokenizer.model_max_length))
    top = Counter(tok_ids).most_common(20)
    top_tokens = [(tokenizer.convert_ids_to_tokens([tid])[0], cnt) for tid, cnt in top]
    metrics_history['top_tokens'] = top_tokens
    # compute small-sample rouge/bleu and generation lengths
    small_val = val_dataset.select(range(min(50, len(val_dataset))))
    gen_lens = []
    try:
        gen_metrics = compute_generation_metrics(model, tokenizer, small_val, batch=2)
        metrics_history['rouge'] = gen_metrics['rouge']
        metrics_history['bleu'] = gen_metrics['bleu']
        # collect gen lengths
        for i in range(0, len(small_val), 2):
            be = small_val[i:i+2]
            texts = [ex['text'] for ex in be]
            inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True).to(next(model.parameters()).device)
            with torch.no_grad():
                outs = model.generate(**inputs, max_new_tokens=128, do_sample=False)
            dec = tokenizer.batch_decode(outs, skip_special_tokens=True)
            gen_lens.extend([len(tokenizer(x)['input_ids']) for x in dec])
        metrics_history['gen_lens'] = gen_lens
    except Exception as e:
        print('Metric computation failed:', e)
    if torch.cuda.is_available():
        metrics_history['gpu_mem'] = [torch.cuda.memory_allocated(0) / 1024**3]
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, 'post_training_metrics.json'), 'w', encoding='utf-8') as f:
        json.dump(metrics_history, f, ensure_ascii=False, indent=2)
    plot_and_save(metrics_history, output_dir)

# Usage: after training, run:
# collect_and_plot_post_training(trainer, model, tokenizer, train_dataset, val_dataset, test_dataset, CONFIG['output_dir'])